# Proyección de cierre 2026 · consumo, saldo e inventario por centro y situación (TVL y TATSA)

Responde, para cada **Empresa → Centro de costo → Situación presupuestaria** visible en Odoo:
1. **¿Cuánto voy a consumir?** Octubre real, noviembre (real a la fecha + proyección) y diciembre proyectado.
2. **¿Con qué lo cubro?** Primero el **inventario**, luego lo que ya está **en tránsito** (RQ con OC y sin OC) y lo que falta es **compra nueva**.
3. **¿Me alcanza el saldo?** Solo la compra nueva gasta saldo: **Saldo al 31‑DIC = saldo real de hoy − compra nueva − ajustes**.
4. **¿Baja el inventario?** Inventario al inicio de octubre, hoy y al 31‑DIC.

**Cómo se trabaja:**
- El notebook genera **`Ajustes_Proyeccion_2026.xlsx`**, que es lo **único que se edita** (columnas amarillas):
  - `AJUSTES_GRUPO`: un monto en soles por grupo y mes (gasto extra o menos que no está en el cálculo).
  - `REPUESTOS`: corrección puntual de la cantidad de un repuesto por mes. Vienen los A y los B‑Z; se pueden agregar filas con cualquier llave o código.
  - `SERVICIOS`: corrección del monto de un servicio por mes.
- Al volver a ejecutar, el notebook lee los ajustes, los conserva (respaldo `*_respaldo.xlsx`) y recalcula **`Proyeccion_Presupuesto_2026.xlsx`**:
  - `RESUMEN`: una fila por grupo con subtotales por centro y empresa.
  - `FLUJO_MENSUAL`: saldo e inventario mes a mes.
  - `SEGUIMIENTO_MENSUAL`: gasto de enero a diciembre.
  - `DETALLE_REPUESTOS` y `DETALLE_SERVICIOS`: el cálculo por ítem, con ABC‑XYZ.
  - `ACCIONES`: qué comprar si sobra, qué postergar si falta, qué stock consumir y qué no llega al cierre.

**Proyección del consumo:**
- **Repuestos e insumos almacenables:** kardex 91 − 92 (buses de 12, 18 y 9 m, por bus × buses activos) y 10 (consumo interno). Los regulares (X, Y) usan el promedio de los últimos 6 meses cerrados; los erráticos (Z), el de 12 meses; sin consumo en 12 meses, 0.
- **Servicios:** facturas con IGV. Los recurrentes (facturados en 3 o más de los últimos 6 meses) usan su promedio; los eventuales, 0, pero se listan para decidir.
- **Prueba de confianza:** octubre se proyecta solo con datos hasta setiembre y se compara con el real (`Desvio_Proy_OCT`).

**Alcance:** lo visible en Odoo (centro + situación): TVL TR01421 y TR01432; TATSA TA01432. Las situaciones de un mismo SKU y centro se unen como en Odoo. Los repuestos a destruir ya no tienen stock y no se mapean. Se excluye la OC intercompañía `G3370009` (informe de costos), que no es consumo.

**Orden de ejecución:** `Tablas_Actual_BI` (kardex) → `Saldo_Disponible_Real` (saldo, pendientes y gasto por mes) → este notebook. Montos en soles con IGV.

##### Librerías

In [ ]:
import xmlrpc.client
import os
import re
import shutil
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openpyxl.styles import Font, PatternFill, Alignment, Border
from openpyxl.utils import get_column_letter

##### Parámetros

In [ ]:
ANIO = 2026
MESES_CIERRE = [10, 11, 12]                     # Periodo que se proyecta (desde la ampliación de octubre)
EMPRESAS = {"TVL": 1, "TATSA": 2}
USUARIO = "JIM"                                 # LUIS no puede leer presupuesto
CENTROS_INCLUIDOS = {"TVL": ["TR01421", "TR01432"],
                     "TATSA": ["TA01432"]}      # TA03432 / TA04432: otro proyecto, no se mapean por ahora
PRODUCTOS_EXCLUIDOS = {"G3370009"}              # OC intercompañía (informe de costos): no es consumo

FLOTAS = ["12 METROS", "18 METROS", "9 METROS"]
BUSES_ACTIVOS = {}                              # Baja de flota: {("18 METROS", 12): 70} = 70 buses de 18 m en diciembre
HISTORIA_DESDE = "2024-01-01"
MESES_REGULAR = 6                               # X / Y: promedio de los últimos 6 meses cerrados
MESES_ERRATICO = 12                             # Z: promedio de 12 meses
MESES_ABC = 12
ABC_CORTES = (0.80, 0.95)                       # A <= 80 % del valor, B <= 95 %
XYZ_CORTES = (0.50, 1.00)                       # CV mensual: X < 0,5, Y <= 1,0, Z > 1,0
NIVEL_SERVICIO_Z = 1.65                         # Stock de seguridad y punto de reposición (solo para comprar con sobrante)
SERV_MESES, SERV_MIN_MESES = 6, 3               # Servicio recurrente: facturado en 3 o más de los últimos 6 meses
MESES_SOBRESTOCK = 6                            # Inventario final que cubre más de 6 meses = sobrestock
UMBRAL_JUSTO = 100.0                            # |saldo final| menor a S/ 100 = justo

ARCHIVO_KARDEX = "Información_Proyecto_BI/Fact_Kardex.parquet"
ARCHIVO_FLOTA = "Información_Proyecto_BI/Dim_Flota.parquet"
ARCHIVO_LLAVES = "Bd_Llaves_Actualizado.xlsx"
ARCHIVO_SALDO = "Saldo_Disponible_Real_2024_2026.xlsx"
ARCHIVO_AJUSTES = f"Ajustes_Proyeccion_{ANIO}.xlsx"      # Lo único que se edita (columnas amarillas)
ARCHIVO_EXCEL = f"Proyeccion_Presupuesto_{ANIO}.xlsx"    # Resultado
EXPORTAR_EXCEL = True

NOMBRE_MES = {1: "ENE", 2: "FEB", 3: "MAR", 4: "ABR", 5: "MAY", 6: "JUN", 7: "JUL", 8: "AGO", 9: "SET", 10: "OCT", 11: "NOV", 12: "DIC"}
T = NOMBRE_MES
HOY = pd.Timestamp.today().normalize()
MES_ACTUAL = HOY.month if HOY.year == ANIO else 13
MESES_REALES = [m for m in MESES_CIERRE if m < MES_ACTUAL]              # cerrados: real
MESES_FUTUROS = [m for m in MESES_CIERRE if m >= MES_ACTUAL]            # mes en curso (real a hoy + proyección) y siguientes
ULTIMO_CERRADO = pd.Period(f"{ANIO}-{min(MES_ACTUAL, 13) - 1:02d}", "M") if MES_ACTUAL <= 12 else pd.Period(f"{ANIO}-12", "M")
FIN_ANIO = pd.Timestamp(f"{ANIO}-12-31")
print("Meses reales:", [T[m] for m in MESES_REALES], "| a proyectar:", [T[m] for m in MESES_FUTUROS], "| último mes cerrado:", ULTIMO_CERRADO)

##### Conexión

In [ ]:
load_dotenv()
url, db = os.getenv("URL_ODOO"), os.getenv("BD_ODOO")
username, password = os.getenv(f"USER_{USUARIO}"), os.getenv(f"PASSWORD_{USUARIO}")
uid = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common").authenticate(db, username, password, {})
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)
print(f"uid conectado : {uid}")


def odoo(modelo, metodo, argumentos, opciones=None, empresas=None):
    opciones = dict(opciones or {})
    contexto = {"lang": "es_PE", "tz": "America/Lima", **opciones.pop("context", {})}
    if empresas:
        contexto["allowed_company_ids"] = empresas
    opciones["context"] = contexto
    return models.execute_kw(db, uid, password, modelo, metodo, argumentos, opciones)


def leer_todo(modelo, dominio, campos, empresas, lote=5000, contexto=None):
    filas, desplazamiento = [], 0
    while True:
        bloque = odoo(modelo, "search_read", [dominio],
                      {"fields": campos, "limit": lote, "offset": desplazamiento, "order": "id", "context": contexto or {}}, empresas)
        filas += bloque
        if len(bloque) < lote:
            return filas
        desplazamiento += lote


def leer_ids(modelo, ids, campos, empresas, lote=2000):
    ids, filas = list(ids), []
    for i in range(0, len(ids), lote):
        filas += odoo(modelo, "read", [ids[i:i + lote]], {"fields": campos}, empresas)
    return filas


def m2o_id(valor):
    return valor[0] if isinstance(valor, list) and valor else None


def m2o_nombre(valor):
    return valor[1] if isinstance(valor, list) and len(valor) == 2 else None

##### Maestros: productos y llaves, unidades y flota

In [ ]:
llaves = pd.read_excel(ARCHIVO_LLAVES, engine="calamine")
llaves = llaves[["Cod_Producto", "Cod_Grupo_Sku", "Producto_Grupo"]].dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto")
productos = leer_todo("product.product", [], ["default_code", "name", "detailed_type", "categ_id", "uom_id"], [1, 2], contexto={"active_test": False})
df_productos = pd.DataFrame([{
    "product_id": p["id"], "Cod_Producto": p["default_code"] or None, "Producto": p["name"], "Tipo_Producto": p["detailed_type"],
    "Familia": (m2o_nombre(p["categ_id"]) or "").split("/")[0].strip() or None, "udm_producto": m2o_id(p["uom_id"]),
    "UdM": m2o_nombre(p["uom_id"]),
} for p in productos]).merge(llaves, on="Cod_Producto", how="left")
df_productos["Cod_Grupo_Sku"] = df_productos["Cod_Grupo_Sku"].fillna(df_productos["Cod_Producto"])
df_productos["Producto_Grupo"] = df_productos["Producto_Grupo"].fillna(df_productos["Producto"])
PRODUCTO = df_productos.set_index("product_id").to_dict("index")
POR_CODIGO = df_productos.dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto").set_index("Cod_Producto")
GRUPO_DE_CODIGO = POR_CODIGO["Cod_Grupo_Sku"].to_dict()
ALMACENABLE = (POR_CODIGO["Tipo_Producto"] == "product").to_dict()
INFO_LLAVE = df_productos.drop_duplicates("Cod_Grupo_Sku").set_index("Cod_Grupo_Sku")[["Producto_Grupo", "Familia", "UdM"]]

UDM = {u["id"]: {"factor": u["factor"] or 1.0, "categoria": m2o_id(u["category_id"])}
       for u in odoo("uom.uom", "search_read", [[]], {"fields": ["factor", "category_id"], "context": {"active_test": False}})}


def a_udm_producto(cantidad, udm_origen, product_id):
    destino = PRODUCTO.get(product_id, {}).get("udm_producto")
    o, d = UDM.get(udm_origen), UDM.get(destino)
    if not o or not d or udm_origen == destino or o["categoria"] != d["categoria"]:
        return cantidad
    return cantidad / o["factor"] * d["factor"]


df_flota = pd.read_parquet(ARCHIVO_FLOTA)
df_flota = df_flota[df_flota["Tamaño_Bus"].isin(FLOTAS)]
BUSES_HOY = df_flota.groupby("Tamaño_Bus")["Cod_Bus"].nunique().to_dict()
BUSES_MES = {(f, m): BUSES_ACTIVOS.get((f, m), BUSES_HOY.get(f, 0)) for f in FLOTAS for m in range(1, 13)}
print(f"Productos {len(df_productos):,} | buses por flota: {BUSES_HOY}")

##### 1. Grupos, saldo real y gasto por mes

Del Excel del saldo (`SALDO` y `GASTO_MES`, año 2026, visibles en Odoo y centros incluidos):
- **Grupo** = Empresa + Cuenta analítica + Situación presupuestaria (unida como en Odoo si el SKU tiene varias).
- **Saldo real hoy** = Presupuesto − (Compras + RQ con OC + RQ sin OC + Gasto sin OC). **Saldo Odoo** = lo que muestra la redistribución.
- Un **repuesto (llave)** va al grupo de su SKU con más presupuesto en la empresa. Un **servicio** va al grupo de su SKU + centro.
- Se leen de Odoo las líneas de presupuesto visibles **también las de monto 0** (el saldo no las lista): un repuesto con línea en 0 que se consume igual es demanda de su grupo. Los nombres de grupo se arman igual que en el saldo (control: `del saldo con otro grupo` debe ser 0).
- Grupos sin presupuesto, saldo, consumo ni inventario no se muestran.

In [ ]:
def cod_centro(cuenta):
    m = re.search(r"\[(\w+)\]", cuenta or "")
    return m.group(1) if m else None


def filtrar_alcance(df):
    df = df[(df["Periodo"] == ANIO) & (df["Visible_Odoo"] == "Sí") & ~df["Cod_Producto"].isin(PRODUCTOS_EXCLUIDOS)].copy()
    df["Cod_Centro"] = df["Cuenta Analítica"].map(cod_centro)
    df = df[[c in CENTROS_INCLUIDOS.get(e, []) for e, c in zip(df["Empresa"], df["Cod_Centro"])]].copy()
    df = df.rename(columns={"Situación presupuestaria": "Situación"})
    df["Llave"] = df["Cod_Producto"].map(GRUPO_DE_CODIGO).fillna(df["Cod_Producto"])
    df["Almacenable"] = df["Cod_Producto"].map(ALMACENABLE).fillna(False).astype(bool)
    return df


GRUPO = ["Empresa", "Cuenta Analítica", "Situación"]
saldo_x = filtrar_alcance(pd.read_excel(ARCHIVO_SALDO, sheet_name="SALDO", engine="calamine"))
gasto_mes = filtrar_alcance(pd.read_excel(ARCHIVO_SALDO, sheet_name="GASTO_MES", engine="calamine"))
oc_pend = pd.read_excel(ARCHIVO_SALDO, sheet_name="OC_PENDIENTE_ENTREGA", engine="calamine")
rq_soc = pd.read_excel(ARCHIVO_SALDO, sheet_name="RQ_SIN_OC", engine="calamine")

SALDO_GRUPO = saldo_x.groupby(GRUPO)[["Presupuesto", "Compras", "Gasto_Sin_OC", "RQ_COC", "RQ_SOC", "Saldo", "Disponible_Odoo"]].sum()
GASTO_REAL_MES = gasto_mes.groupby(GRUPO + ["Mes"])["Gasto"].sum()

def lineas_presupuesto(n, empresa_id):
    """SKU + centro con línea de presupuesto del año (también las de monto 0), visibles en Odoo y del alcance.
       La situación se une como en el saldo (todas las situaciones del SKU + centro, en orden alfabético)."""
    dominio = [["company_id", "=", empresa_id], ["crossovered_budget_id.periodo_id.fiscal_year_id.name", "=", str(ANIO)],
               ["crossovered_budget_id.state", "not in", ["cancel", "draft"]]]
    grupos = odoo("crossovered.budget.lines", "read_group",
                  [dominio, ["planned_amount"], ["product_id", "analytic_account_id", "general_budget_id"]], {"lazy": False}, [empresa_id])
    ids = odoo("crossovered.budget.lines", "search", [dominio], {}, [empresa_id])
    visibles = {(m2o_id(l["product_id"]), m2o_id(l["analytic_account_id"]), m2o_id(l["general_budget_id"]))
                for l in leer_ids("crossovered.budget.lines", ids, ["product_id", "analytic_account_id", "general_budget_id"], [empresa_id])}
    df = pd.DataFrame([{"product_id": m2o_id(g["product_id"]), "Cuenta Analítica": m2o_nombre(g["analytic_account_id"]),
                        "Sit": m2o_nombre(g["general_budget_id"]), "Ppto": g["planned_amount"] or 0.0,
                        "Visible": (m2o_id(g["product_id"]), m2o_id(g["analytic_account_id"]), m2o_id(g["general_budget_id"])) in visibles}
                       for g in grupos])
    df = df.dropna(subset=["product_id", "Cuenta Analítica", "Sit"])
    df["Cod_Producto"] = df["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Cod_Producto"))
    df = df[np.array([cod_centro(c) in CENTROS_INCLUIDOS.get(n, []) for c in df["Cuenta Analítica"]]) & ~df["Cod_Producto"].isin(PRODUCTOS_EXCLUIDOS).values]
    out = df.groupby(["Cod_Producto", "Cuenta Analítica"]).agg(
        Situación=("Sit", lambda s: " + ".join(sorted(set(s)))), Presupuesto=("Ppto", "sum"), Visible=("Visible", "any")).reset_index()
    out = out[out["Visible"]].drop(columns="Visible").assign(Empresa=n)
    out["Llave"] = out["Cod_Producto"].map(GRUPO_DE_CODIGO).fillna(out["Cod_Producto"])
    out["Almacenable"] = out["Cod_Producto"].map(ALMACENABLE).fillna(False).astype(bool)
    return out


lineas = pd.concat([lineas_presupuesto(n, e) for n, e in EMPRESAS.items()], ignore_index=True)
claves_saldo = set(zip(saldo_x["Empresa"], saldo_x["Cod_Producto"], saldo_x["Cuenta Analítica"], saldo_x["Situación"]))
con_ppto_saldo = saldo_x[saldo_x["Presupuesto"] != 0]
distintas = [k for k in zip(con_ppto_saldo["Empresa"], con_ppto_saldo["Cod_Producto"], con_ppto_saldo["Cuenta Analítica"], con_ppto_saldo["Situación"])
             if k not in set(zip(lineas["Empresa"], lineas["Cod_Producto"], lineas["Cuenta Analítica"], lineas["Situación"]))]
print(f"Líneas de presupuesto visibles (SKU + centro, incluidas las de monto 0): {len(lineas):,} | del saldo con otro grupo: {len(distintas)}")

# Grupo de cada llave almacenable: el de su SKU con más presupuesto en la empresa (una línea en 0 también cuenta: su consumo es del grupo)
GRUPO_LLAVE = {}
for (n, ll), g in lineas[lineas["Almacenable"]].groupby(["Empresa", "Llave"]):
    fila = g.groupby(["Cuenta Analítica", "Situación"])["Presupuesto"].sum().idxmax()
    GRUPO_LLAVE[(n, ll)] = (n, *fila)
LLAVES = {n: {ll for (e, ll) in GRUPO_LLAVE if e == n} for n in EMPRESAS}

# Servicios y bienes no almacenables: cada SKU + centro con línea de presupuesto o con gasto en el saldo
claves_srv = GRUPO + ["Cod_Producto"]
SERVICIOS = pd.concat([lineas[~lineas["Almacenable"]][claves_srv], saldo_x[~saldo_x["Almacenable"]][claves_srv]]).drop_duplicates()
SERVICIOS = SERVICIOS.merge(saldo_x.groupby(claves_srv)[["Presupuesto", "RQ_COC", "RQ_SOC"]].sum().reset_index(), on=claves_srv, how="left").fillna(
    {"Presupuesto": 0.0, "RQ_COC": 0.0, "RQ_SOC": 0.0})
SERVICIOS["Producto"] = SERVICIOS["Cod_Producto"].map(POR_CODIGO["Producto"])
print(SALDO_GRUPO[["Presupuesto", "Saldo", "Disponible_Odoo"]].round(0).to_string())
print(f"Llaves almacenables: { {n: len(v) for n, v in LLAVES.items()} } | servicios (SKU + centro): {len(SERVICIOS):,}")

##### 2. Precio, lead time, consumo del kardex y ABC‑XYZ

- **Precio:** última compra de la llave en soles con IGV (OC confirmadas); si no hay, costo del kardex × 1,18. Con ese precio se valoriza consumo e inventario.
- **Lead time:** mediana de días entre el RQ y la primera recepción de la llave.
- **Consumo:** salidas a buses 91 − 92 (de cualquier almacén: la llave de buses la cubre TATSA si tiene presupuesto allí, si no TVL) y consumo interno 10 del almacén de la empresa (si la llave tiene presupuesto en esa empresa).
- **ABC** por valor de consumo de los últimos 12 meses cerrados; **XYZ** por CV mensual de todo el histórico.

In [ ]:
def leer_oc(empresa_id):
    ordenes = {o["id"]: o for o in leer_todo("purchase.order",
               [["company_id", "=", empresa_id], ["state", "in", ["purchase", "done"]], ["date_order", ">=", HISTORIA_DESDE]],
               ["name", "currency_rate", "date_approve", "date_order", "requisition_id"], [empresa_id])}
    ids_rq = list({m2o_id(o["requisition_id"]) for o in ordenes.values() if o["requisition_id"]})
    rqs = {r["id"]: r["create_date"][:10] for r in leer_todo("purchase.requisition", [["id", "in", ids_rq]], ["create_date"], [empresa_id])}
    lineas = leer_todo("purchase.order.line", [["order_id", "in", list(ordenes)], ["product_id", "!=", False]],
                       ["order_id", "product_id", "product_uom", "product_qty", "price_total"], [empresa_id])
    filas = []
    for l in lineas:
        o, p = ordenes[m2o_id(l["order_id"])], m2o_id(l["product_id"])
        cant = a_udm_producto(l["product_qty"] or 0.0, m2o_id(l["product_uom"]), p)
        if cant > 0 and l["price_total"]:
            filas.append({"po_line_id": l["id"], "Fecha_OC": (o["date_approve"] or o["date_order"])[:10], "Fecha_RQ": rqs.get(m2o_id(o["requisition_id"])),
                          "Llave": PRODUCTO.get(p, {}).get("Cod_Grupo_Sku"), "precio": l["price_total"] / (o["currency_rate"] or 1.0) / cant})
    return pd.DataFrame(filas)


def primeras_recepciones(empresa_id):
    movs = leer_todo("stock.move", [["company_id", "=", empresa_id], ["state", "=", "done"], ["purchase_line_id", "!=", False],
                                    ["location_id.usage", "=", "supplier"], ["date", ">=", HISTORIA_DESDE]], ["purchase_line_id", "date"], [empresa_id])
    return pd.DataFrame([{"po_line_id": m2o_id(m["purchase_line_id"]), "Fecha": m["date"][:10]} for m in movs]).groupby("po_line_id")["Fecha"].min()


oc = pd.concat([leer_oc(e) for e in EMPRESAS.values()], ignore_index=True).sort_values("Fecha_OC")
recep = pd.concat([primeras_recepciones(e) for e in EMPRESAS.values()])
PRECIO_OC = oc.groupby("Llave")["precio"].last()
lt = oc.dropna(subset=["Fecha_RQ"]).join(recep.rename("Fecha_Recepcion"), on="po_line_id", how="inner")
lt["Dias"] = (pd.to_datetime(lt["Fecha_Recepcion"]) - pd.to_datetime(lt["Fecha_RQ"])).dt.days
LEAD_TIME = lt[lt["Dias"] >= 0].groupby("Llave")["Dias"].median()
LT_GENERAL = lt[lt["Dias"] >= 0]["Dias"].median()

kardex_todo = pd.read_parquet(ARCHIVO_KARDEX)
kardex_todo = kardex_todo[kardex_todo["Fecha"] >= HISTORIA_DESDE].copy()
kardex_todo["Almacen"] = kardex_todo["Empresa"].map({"Tatsa": "TATSA", "Tvl": "TVL"})
kardex_todo["Llave"] = kardex_todo["Cod_Producto"].map(GRUPO_DE_CODIGO).fillna(kardex_todo["Cod_Producto"])

kardex = kardex_todo[kardex_todo["Cod_Movimiento"].isin([91, 92, 10])].copy()
kardex["Cantidad"] = kardex["Cant_Salida"] - kardex["Cant_Ingreso"]
kardex = kardex.merge(df_flota.rename(columns={"Tamaño_Bus": "Flota"}), on="Cod_Bus", how="left")
kardex["Flota"] = np.where(kardex["Cod_Movimiento"] == 10, "No aplica", kardex["Flota"].fillna("Otras"))
kardex["Tipo_Mantto"] = np.where(kardex["Cod_Movimiento"] == 10, "Consumo interno", kardex["Tipo_Mantto"].fillna("Correctivo"))
kardex = kardex[(kardex["Cod_Movimiento"] == 10) | kardex["Flota"].isin(FLOTAS)]


def duena(cod_mov, almacen, llave):
    if cod_mov == 10:
        return almacen if llave in LLAVES[almacen] else None
    return "TATSA" if llave in LLAVES["TATSA"] else "TVL" if llave in LLAVES["TVL"] else None


kardex["Empresa"] = [duena(c, a, ll) for c, a, ll in zip(kardex["Cod_Movimiento"], kardex["Almacen"], kardex["Llave"])]
kardex = kardex[kardex["Empresa"].notna()].copy()
kardex["Periodo_Mes"] = kardex["Fecha"].dt.to_period("M")
costo = kardex_todo[kardex_todo["Cant_Salida"] > 0].groupby("Llave")[["Valor_Salida", "Cant_Salida"]].sum()
PRECIO = PRECIO_OC.combine_first((costo["Valor_Salida"] / costo["Cant_Salida"] * 1.18).clip(lower=0))

MESES_HIST = pd.period_range(pd.Period(HISTORIA_DESDE[:7], "M"), ULTIMO_CERRADO, freq="M")


def demanda(n):
    d = kardex[(kardex["Empresa"] == n) & (kardex["Periodo_Mes"] <= ULTIMO_CERRADO)].groupby(["Llave", "Periodo_Mes"])["Cantidad"].sum()
    idx = pd.MultiIndex.from_product([d.index.get_level_values(0).unique(), MESES_HIST], names=["Llave", "Periodo_Mes"])
    return d.reindex(idx, fill_value=0.0).unstack().clip(lower=0)


def clasificar(n):
    cant = demanda(n)
    m12 = MESES_HIST[-MESES_ABC:]
    valor = cant[m12].sum(axis=1) * cant.index.map(PRECIO).fillna(0.0)
    valor = valor.sort_values(ascending=False)
    part = valor.cumsum() / valor.sum()
    cv = (cant.std(axis=1, ddof=0) / cant.mean(axis=1)).replace([np.inf, -np.inf], np.nan)
    c = pd.DataFrame({"Valor_12M": valor, "CV": cv, "Prom_6M": cant[MESES_HIST[-MESES_REGULAR:]].mean(axis=1),
                      "Prom_12M": cant[m12].mean(axis=1), "Desv_12M": cant[m12].std(axis=1, ddof=0),
                      "Meses_Con_Consumo_12M": (cant[m12] > 0).sum(axis=1),
                      "ABC": pd.Series(np.where(part <= ABC_CORTES[0], "A", np.where(part <= ABC_CORTES[1], "B", "C")), index=valor.index),
                      "XYZ": pd.Series(np.where(cv < XYZ_CORTES[0], "X", np.where(cv <= XYZ_CORTES[1], "Y", "Z")), index=cv.index)})
    c.loc[c["Valor_12M"] <= 0, ["ABC", "XYZ"]] = ["-", "-"]
    tipo = kardex[(kardex["Empresa"] == n) & kardex["Periodo_Mes"].isin(m12)].groupby(["Llave", "Tipo_Mantto"])["Cantidad"].sum().unstack(fill_value=0.0)
    c["Categoria"] = tipo.idxmax(axis=1).map({"Preventivo": "PREVENTIVO", "Correctivo": "CORRECTIVO", "Consumo interno": "CONSUMO INTERNO"})
    return c.assign(Empresa=n)


def proyectar_mes(n, mes, fin, xyz):
    """Unidades del mes `mes` con la historia hasta `fin`: buses = tasa por bus × buses activos del mes; consumo interno = promedio.
       Regulares (X, Y) con 6 meses, erráticos (Z) con 12; sin consumo en los últimos 12 meses = 0."""
    resultado = {}
    for meses, filtro in [(MESES_REGULAR, lambda s: s != "Z"), (MESES_ERRATICO, lambda s: s == "Z")]:
        ventana = pd.period_range(fin - meses + 1, fin, freq="M")
        h = kardex[(kardex["Empresa"] == n) & kardex["Periodo_Mes"].isin(ventana)]
        tasa = h.groupby(["Llave", "Flota"])["Cantidad"].sum() / meses
        v = pd.Series([q if f not in FLOTAS else q / max(BUSES_HOY.get(f, 1), 1) * BUSES_MES[(f, mes)] for (ll, f), q in tasa.items()],
                      index=tasa.index.get_level_values(0), dtype="float64").groupby(level=0).sum().clip(lower=0)
        for ll, q in v.items():
            if filtro(xyz.get(ll, "-")):
                resultado[ll] = q
    return pd.Series(resultado, dtype="float64")


clases = pd.concat([clasificar(n).reset_index().rename(columns={"index": "Llave"}) for n in EMPRESAS], ignore_index=True)
print(clases.groupby(["Empresa", "ABC"]).agg(Llaves=("Llave", "size"), Valor_12M=("Valor_12M", "sum")).round(0).to_string())

##### 3. Stock (hoy y al cierre de cada mes), tránsito y consumo real

- **Stock hoy** de Odoo (ubicaciones internas). El stock al cierre de un mes pasado = stock hoy − movimientos del kardex posteriores.
- **Inventario de un grupo:** stock de sus llaves en el almacén de la empresa. Las llaves de TATSA suman también el stock de TVL si TVL no las presupuesta (el stock de repuestos de buses de TVL cubre la demanda de buses).
- **Tránsito:** RQ con OC (llega en su fecha de entrega prevista; si ya venció, este mes) y RQ sin OC (llega en hoy + lead time). Lo que llega después del 31‑DIC no cubre el cierre.
- **Servicios:** consumo = facturas con IGV por mes; tránsito = sus RQ con OC y sin OC del saldo.

In [ ]:
stock = []
for n, e in EMPRESAS.items():
    stock += [{"Almacen": n, "product_id": m2o_id(q["product_id"]), "Cantidad": q["quantity"]}
              for q in leer_todo("stock.quant", [["company_id", "=", e], ["location_id.usage", "=", "internal"], ["quantity", ">", 0]],
                                 ["product_id", "quantity"], [e])]
df_stock = pd.DataFrame(stock)
df_stock["Llave"] = df_stock["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Cod_Grupo_Sku"))
STOCK_HOY = df_stock.groupby(["Almacen", "Llave"])["Cantidad"].sum()

inicio_cierre = pd.Timestamp(f"{ANIO}-{MESES_CIERRE[0]:02d}-01")
movs = kardex_todo[kardex_todo["Fecha"] >= inicio_cierre].assign(Neto=lambda d: d["Cant_Ingreso"] - d["Cant_Salida"])


def stock_al(fecha):
    """Stock por almacén + llave al final del día `fecha` (stock hoy menos lo que se movió después)."""
    despues = movs[movs["Fecha"] > fecha].groupby(["Almacen", "Llave"])["Neto"].sum()
    return STOCK_HOY.sub(despues, fill_value=0.0).clip(lower=0)


def almacenes(n, ll):
    return ["TVL"] if n == "TVL" else ["TATSA"] + (["TVL"] if ll not in LLAVES["TVL"] else [])


def stock_llave(serie, n, ll):
    return sum(serie.get((a, ll), 0.0) for a in almacenes(n, ll))


CORTES = {"inicio": inicio_cierre - pd.Timedelta(days=1)}
for m in MESES_REALES:
    CORTES[m] = pd.Period(f"{ANIO}-{m:02d}", "M").end_time.normalize()
STOCK_CORTE = {k: stock_al(f) for k, f in CORTES.items()}

k_cierre = kardex[(kardex["Fecha"].dt.year == ANIO) & kardex["Fecha"].dt.month.isin(MESES_CIERRE)]
REAL_UND = k_cierre.assign(Mes=k_cierre["Fecha"].dt.month).groupby(["Empresa", "Llave", "Mes"])["Cantidad"].sum()


def mes_llegada(fecha):
    if pd.isna(fecha) or fecha <= HOY:
        return MES_ACTUAL
    return fecha.month if fecha.year == ANIO else 13          # 13 = después del cierre


for d in (oc_pend, rq_soc):
    d["Llave"] = d["Cod_Producto"].map(GRUPO_DE_CODIGO).fillna(d["Cod_Producto"])
oc_pend["Mes_Llegada"] = pd.to_datetime(oc_pend["Fecha_Entrega_Prevista"], errors="coerce").map(mes_llegada)
rq_soc["Mes_Llegada"] = [mes_llegada(HOY + pd.Timedelta(days=float(LEAD_TIME.get(ll, LT_GENERAL)))) for ll in rq_soc["Llave"]]
TRANSITO = pd.concat([oc_pend.assign(Tipo="RQ_COC"), rq_soc.assign(Tipo="RQ_SOC")]).groupby(["Empresa", "Llave", "Tipo", "Mes_Llegada"])["Cant_Pendiente"].sum()


def ejecucion_servicios(n, empresa_id):
    apuntes = leer_todo("account.move.line",
                        [["company_id", "=", empresa_id], ["parent_state", "=", "posted"], ["product_id", "!=", False],
                         ["exclude_from_invoice_tab", "=", False], ["move_id.move_type", "in", ["in_invoice", "in_refund", "in_receipt"]],
                         ["date", ">=", f"{ANIO - 1}-01-01"], ["date", "<=", f"{ANIO}-12-31"], ["product_id.detailed_type", "!=", "product"]],
                        ["product_id", "analytic_account_id", "balance", "price_subtotal", "price_total", "date"], [empresa_id])
    df = pd.DataFrame([{"Cod_Producto": PRODUCTO.get(m2o_id(a["product_id"]), {}).get("Cod_Producto"), "Cuenta Analítica": m2o_nombre(a["analytic_account_id"]),
                        "Periodo_Mes": pd.Period(a["date"][:7], "M"),
                        "Monto": (a["balance"] or 0.0) * ((a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0)} for a in apuntes])
    return df.assign(Empresa=n)


serv_real = pd.concat([ejecucion_servicios(n, e) for n, e in EMPRESAS.items()], ignore_index=True)
SERV_MES = serv_real.groupby(["Empresa", "Cuenta Analítica", "Cod_Producto", "Periodo_Mes"])["Monto"].sum()
print(f"Stock {len(df_stock):,} quants | tránsito {len(oc_pend):,} OC pendientes + {len(rq_soc):,} RQ sin OC | facturas de servicios {len(serv_real):,}")

##### 4. Ajustes del usuario (`Ajustes_Proyeccion_2026.xlsx`)

Se leen las columnas amarillas del archivo de ajustes (si existe):
- `AJUSTES_GRUPO`: `Ajuste_NOV`, `Ajuste_DIC` en soles por grupo. Se suman a la compra nueva del grupo (gasto adicional si es positivo, menos gasto si es negativo).
- `REPUESTOS`: `Corr_NOV`, `Corr_DIC` en unidades **totales del mes** por llave. Reemplazan la proyección; el mes en curso descuenta lo ya consumido.
- `SERVICIOS`: `Corr_NOV`, `Corr_DIC` en soles por servicio + centro.

Una celda vacía significa "usar el cálculo".

In [ ]:
COL_AJ = {m: f"Ajuste_{T[m]}" for m in MESES_FUTUROS}
COL_CORR = {m: f"Corr_{T[m]}" for m in MESES_FUTUROS}
CLAVE_AJ_GRUPO, CLAVE_AJ_REP, CLAVE_AJ_SRV = GRUPO, ["Empresa", "Llave"], ["Empresa", "Cuenta Analítica", "Cod_Producto"]


def leer_ajustes(hoja, clave, columnas):
    try:
        df = pd.read_excel(ARCHIVO_AJUSTES, sheet_name=hoja, engine="calamine")
    except Exception:
        return pd.DataFrame(columns=clave + columnas + ["Comentario"])
    if hoja == "REPUESTOS" and "Llave" in df:
        df["Llave"] = [GRUPO_DE_CODIGO.get(x, x) if isinstance(x, str) else x for x in df["Llave"].astype("object")]
    for c in columnas:
        df[c] = pd.to_numeric(df[c], errors="coerce") if c in df else np.nan
    df["Comentario"] = df["Comentario"] if "Comentario" in df else np.nan
    df = df.dropna(subset=clave)
    df = df[df[columnas].notna().any(axis=1) | df["Comentario"].notna()]
    return df[clave + columnas + ["Comentario"]].drop_duplicates(clave, keep="last")


aj_grupo = leer_ajustes("AJUSTES_GRUPO", CLAVE_AJ_GRUPO, list(COL_AJ.values()))
aj_rep = leer_ajustes("REPUESTOS", CLAVE_AJ_REP, list(COL_CORR.values()))
aj_srv = leer_ajustes("SERVICIOS", CLAVE_AJ_SRV, list(COL_CORR.values()))
print(f"Ajustes leídos: grupos {len(aj_grupo)} | repuestos {len(aj_rep)} | servicios {len(aj_srv)}")

##### 5. Proyección por repuesto y servicio, y cobertura mes a mes

Para cada llave, mes a mes (mes en curso: solo lo que falta consumir):
1. Consumo del mes = corrección del usuario, si hay; si no, la proyección.
2. Se cubre primero con el **inventario**, luego con el **tránsito** que llega ese mes o antes, y el resto es **compra nueva** (gasta saldo ese mes).
3. Inventario al cierre del mes = inventario + llegadas + compra nueva − consumo.

**Estado:** COMPRA NECESARIA · CUBIERTO · SOBRESTOCK (el inventario al 31‑DIC cubre más de 6 meses) · SIN ROTACIÓN (stock sin consumo en 12 meses) · SIN CONSUMO. **Riesgo:** compra nueva de una llave cuyo lead time no alcanza para llegar al 31‑DIC.

In [ ]:
DIAS_AL_CIERRE = (FIN_ANIO - HOY).days
filas = []
for (n, ll), grupo in GRUPO_LLAVE.items():
    filas.append({"Empresa": n, "Cuenta Analítica": grupo[1], "Situación": grupo[2], "Llave": ll})
rep = pd.DataFrame(filas).merge(clases, on=["Empresa", "Llave"], how="left")
rep[["Producto", "Familia", "UdM"]] = INFO_LLAVE.reindex(rep["Llave"]).values
rep["Precio"] = rep["Llave"].map(PRECIO).fillna(0.0)
for c in ["Valor_12M", "Prom_6M", "Prom_12M", "Desv_12M", "Meses_Con_Consumo_12M"]:
    rep[c] = rep[c].fillna(0.0)
rep[["ABC", "XYZ"]] = rep[["ABC", "XYZ"]].fillna("-")
rep["ABC_XYZ"] = np.where(rep["ABC"] == "-", "SIN CONSUMO", rep["ABC"] + "-" + rep["XYZ"])
rep["Categoria"] = rep["Categoria"].fillna("SIN CONSUMO")
rep["Metodo"] = np.select([rep["Prom_12M"] <= 0, rep["XYZ"] == "Z"],
                          ["Sin consumo en 12 meses: 0", f"Errático: promedio {MESES_ERRATICO} meses"], default=f"Regular: promedio {MESES_REGULAR} meses")
rep.loc[rep["ABC"].isin(["A", "B"]) & (rep["XYZ"] == "Z"), "Metodo"] += " · REVISAR"
rep["Lead_Time_Dias"] = rep["Llave"].map(LEAD_TIME).fillna(LT_GENERAL)
rep["Stock_Hoy"] = [stock_llave(STOCK_HOY, n, ll) for n, ll in zip(rep["Empresa"], rep["Llave"])]
for k in STOCK_CORTE:
    rep[f"Stock_{'Inicio' if k == 'inicio' else T[k]}"] = [stock_llave(STOCK_CORTE[k], n, ll) for n, ll in zip(rep["Empresa"], rep["Llave"])]
for tipo in ["RQ_COC", "RQ_SOC"]:
    t = TRANSITO.xs(tipo, level="Tipo").groupby(level=[0, 1]).sum()
    rep[tipo] = [t.get((n, ll), 0.0) for n, ll in zip(rep["Empresa"], rep["Llave"])]

# Real de los meses del cierre y proyección (prueba con los meses reales; proyección de los futuros)
XYZ = {n: rep[rep["Empresa"] == n].set_index("Llave")["XYZ"].to_dict() for n in EMPRESAS}
for m in MESES_CIERRE:
    if m <= MES_ACTUAL:
        rep[f"Real_{T[m]}"] = [REAL_UND.get((n, ll, m), 0.0) for n, ll in zip(rep["Empresa"], rep["Llave"])]
for m in MESES_CIERRE:
    fin = pd.Period(f"{ANIO}-{m:02d}", "M") - 1 if m in MESES_REALES else ULTIMO_CERRADO
    proy = {n: proyectar_mes(n, m, fin, XYZ[n]) for n in EMPRESAS}
    rep[f"{'Prueba' if m in MESES_REALES else 'Sug'}_{T[m]}"] = [proy[n].get(ll, 0.0) for n, ll in zip(rep["Empresa"], rep["Llave"])]
rep.loc[rep["Prom_12M"] <= 0, [f"Sug_{T[m]}" for m in MESES_FUTUROS]] = 0.0

# Correcciones puntuales del usuario
rep = rep.merge(aj_rep.rename(columns={"Comentario": "Comentario_Usuario"}), on=CLAVE_AJ_REP, how="left")
for m in MESES_FUTUROS:
    rep[f"Proy_{T[m]}"] = rep[COL_CORR[m]].fillna(rep[f"Sug_{T[m]}"])

# Cobertura mes a mes
arr = {(n, ll, mes): q for (n, ll, _, mes), q in TRANSITO.items()}
sim = []
for r in rep.itertuples(index=False):
    inv = r.Stock_Hoy
    fila = {}
    pendientes = {mes: q for (n, ll, mes), q in arr.items() if n == r.Empresa and ll == r.Llave}
    for m in MESES_FUTUROS:
        llega = sum(q for mes, q in pendientes.items() if mes <= m)
        for mes in [x for x in pendientes if x <= m]:
            pendientes.pop(mes)
        proy = getattr(r, f"Proy_{T[m]}")
        cons = max(proy - getattr(r, f"Real_{T[m]}"), 0.0) if m == MES_ACTUAL else proy
        usa_inv = min(cons, inv)
        usa_tra = min(cons - usa_inv, llega)
        compra = cons - usa_inv - usa_tra
        inv = inv - usa_inv + llega - usa_tra
        fila.update({f"Pend_{T[m]}": cons, f"Usa_Inv_{T[m]}": usa_inv, f"Usa_Tra_{T[m]}": usa_tra, f"Compra_{T[m]}": compra,
                     f"Llega_{T[m]}": llega, f"Inv_Fin_{T[m]}": inv})
    fila["Transito_Post_Cierre"] = sum(pendientes.values())
    fila["Inv_31DIC"] = inv
    sim.append(fila)
rep = pd.concat([rep, pd.DataFrame(sim, index=rep.index)], axis=1)

COMPRA_UND = rep[[f"Compra_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
rep["Compra_Nueva_Und"] = COMPRA_UND
rep["Compra_Nueva"] = COMPRA_UND * rep["Precio"]
prom = rep[[f"Proy_{T[m]}" for m in MESES_FUTUROS]].mean(axis=1) if MESES_FUTUROS else rep["Prom_6M"]
rep["Cobertura_31DIC_Meses"] = np.where(prom > 0, rep["Inv_31DIC"] / prom.where(prom > 0, 1), np.where(rep["Inv_31DIC"] > 0, np.inf, 0.0))
rep["Estado"] = np.select(
    [(rep["Prom_12M"] <= 0) & (rep["Stock_Hoy"] > 0), rep["Prom_12M"] <= 0, rep["Compra_Nueva_Und"] > 1e-9, rep["Cobertura_31DIC_Meses"] > MESES_SOBRESTOCK],
    ["SIN ROTACIÓN", "SIN CONSUMO", "COMPRA NECESARIA", "SOBRESTOCK"], default="CUBIERTO")
rep["Riesgo_Cierre"] = np.where((rep["Compra_Nueva_Und"] > 1e-9) & (rep["Lead_Time_Dias"] > DIAS_AL_CIERRE),
                                "NO LLEGA AL 31-DIC (lead time " + rep["Lead_Time_Dias"].round(0).astype(int).astype(str) + " días)", "")
rep["Stock_Seguridad"] = NIVEL_SERVICIO_Z * rep["Desv_12M"] * np.sqrt(rep["Lead_Time_Dias"] / 30)
rep["Punto_Reposicion"] = rep["Prom_6M"].where(rep["XYZ"] != "Z", rep["Prom_12M"]) * rep["Lead_Time_Dias"] / 30 + rep["Stock_Seguridad"]

"""
    SERVICIOS: CONSUMO = FACTURAS; RECURRENTE SI SE FACTURÓ EN 3 O MÁS DE LOS ÚLTIMOS 6 MESES
"""


def promedio_servicio(n, cta, cod, fin):
    ventana = pd.period_range(fin - SERV_MESES + 1, fin, freq="M")
    v = [SERV_MES.get((n, cta, cod, p), 0.0) for p in ventana]
    meses = sum(x > 0 for x in v)
    return (sum(v) / SERV_MESES if meses >= SERV_MIN_MESES else 0.0), meses


srv = SERVICIOS.copy()
for m in MESES_CIERRE:
    if m <= MES_ACTUAL:
        srv[f"Real_{T[m]}"] = [SERV_MES.get((n, c, p, pd.Period(f"{ANIO}-{m:02d}", "M")), 0.0)
                               for n, c, p in zip(srv["Empresa"], srv["Cuenta Analítica"], srv["Cod_Producto"])]
srv["Ejecutado_Anio"] = [sum(v for (e, c, p, per), v in SERV_MES.items() if (e, c, p) == k and per.year == ANIO)
                         for k in zip(srv["Empresa"], srv["Cuenta Analítica"], srv["Cod_Producto"])]
base = [promedio_servicio(n, c, p, ULTIMO_CERRADO) for n, c, p in zip(srv["Empresa"], srv["Cuenta Analítica"], srv["Cod_Producto"])]
srv["Prom_6M"], srv["Meses_Facturados_6M"] = [b[0] for b in base], [b[1] for b in base]
srv["Tipo"] = np.where(srv["Meses_Facturados_6M"] >= SERV_MIN_MESES, "RECURRENTE",
                       np.where(srv["Ejecutado_Anio"] != 0, "EVENTUAL: decidir", "SIN EJECUCIÓN: decidir"))
for m in MESES_REALES:
    fin = pd.Period(f"{ANIO}-{m:02d}", "M") - 1
    srv[f"Prueba_{T[m]}"] = [promedio_servicio(n, c, p, fin)[0] for n, c, p in zip(srv["Empresa"], srv["Cuenta Analítica"], srv["Cod_Producto"])]
for m in MESES_FUTUROS:
    srv[f"Sug_{T[m]}"] = srv["Prom_6M"]
srv = srv.merge(aj_srv.rename(columns={"Comentario": "Comentario_Usuario"}), on=CLAVE_AJ_SRV, how="left")
srv["Transito"] = srv["RQ_COC"].clip(lower=0) + srv["RQ_SOC"].clip(lower=0)
tra = srv["Transito"].copy()
for m in MESES_FUTUROS:
    srv[f"Proy_{T[m]}"] = srv[COL_CORR[m]].fillna(srv[f"Sug_{T[m]}"])
    cons = (srv[f"Proy_{T[m]}"] - srv[f"Real_{T[m]}"]).clip(lower=0) if m == MES_ACTUAL else srv[f"Proy_{T[m]}"]
    usa = np.minimum(cons, tra)
    srv[f"Pend_{T[m]}"], srv[f"Usa_Tra_{T[m]}"], srv[f"Compra_{T[m]}"] = cons, usa, cons - usa
    tra = tra - usa
srv["Compra_Nueva"] = srv[[f"Compra_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
print(rep.groupby(["Empresa", "Estado"]).agg(Llaves=("Llave", "size"), Compra_Nueva=("Compra_Nueva", "sum")).round(0).to_string())
print(srv.groupby(["Empresa", "Tipo"]).agg(Servicios=("Cod_Producto", "size"), Compra_Nueva=("Compra_Nueva", "sum")).round(0).to_string())

##### 6. Resumen por grupo: consumo vs saldo vs inventario

Una fila por **Empresa → Centro → Situación**, con subtotales:
- `Saldo_Odoo` (lo que deja redistribuir Odoo) y `Saldo_Real_Hoy` (descuenta RQ en tránsito).
- `Inv_Inicio_OCT`, `Inv_Hoy`, `Inv_31DIC`: inventario valorizado del grupo.
- `Consumo_OCT_Real` frente a `Consumo_OCT_Prueba` (proyectado con datos hasta setiembre) → `Desvio_Proy_OCT` (+ = la proyección se pasó, − = se quedó corta; si es grande, conviene ajustar el grupo).
- `Consumo_NOV` (real a la fecha + lo que falta) y `Consumo_DIC`. `Consumo_Pendiente` = lo que falta consumir hasta el 31‑DIC.
- El pendiente se cubre con `Cubre_Inventario` + `Cubre_Transito` + `Compra_Nueva`. Con los `Ajustes`, da el `Saldo_31DIC` = Saldo_Real_Hoy − Compra_Nueva − Ajustes.
- **Estado:** SOBRA (saldo para comprar útil o redistribuir), FALTA (pedir ampliación o postergar) o JUSTO.

In [ ]:
def valor_rep(df, col, und=True):
    return df[col] * df["Precio"] if und else df[col]


def por_grupo(df, columnas):
    return df.groupby(GRUPO)[columnas].sum() if len(df) else pd.DataFrame(columns=columnas)


r_val, s_val = pd.DataFrame(rep[GRUPO]), pd.DataFrame(srv[GRUPO])
r_val["Inv_Inicio"] = valor_rep(rep, "Stock_Inicio")
r_val["Inv_Hoy"] = valor_rep(rep, "Stock_Hoy")
r_val["Inv_31DIC"] = valor_rep(rep, "Inv_31DIC")
r_val["Transito_Post_Cierre"] = valor_rep(rep, "Transito_Post_Cierre")
for m in MESES_REALES:
    r_val[f"Consumo_{T[m]}_Real"], s_val[f"Consumo_{T[m]}_Real"] = valor_rep(rep, f"Real_{T[m]}"), srv[f"Real_{T[m]}"]
    r_val[f"Consumo_{T[m]}_Prueba"], s_val[f"Consumo_{T[m]}_Prueba"] = valor_rep(rep, f"Prueba_{T[m]}"), srv[f"Prueba_{T[m]}"]
    r_val[f"Inv_Fin_{T[m]}"] = valor_rep(rep, f"Stock_{T[m]}")
for m in MESES_FUTUROS:
    real_hoy_r = valor_rep(rep, f"Real_{T[m]}") if m == MES_ACTUAL else 0.0
    real_hoy_s = srv[f"Real_{T[m]}"] if m == MES_ACTUAL else 0.0
    r_val[f"Consumo_{T[m]}"] = real_hoy_r + valor_rep(rep, f"Pend_{T[m]}")
    s_val[f"Consumo_{T[m]}"] = real_hoy_s + srv[f"Pend_{T[m]}"]
    r_val[f"Pend_{T[m]}"], s_val[f"Pend_{T[m]}"] = valor_rep(rep, f"Pend_{T[m]}"), srv[f"Pend_{T[m]}"]
    r_val[f"Usa_Inv_{T[m]}"] = valor_rep(rep, f"Usa_Inv_{T[m]}")
    r_val[f"Usa_Tra_{T[m]}"], s_val[f"Usa_Tra_{T[m]}"] = valor_rep(rep, f"Usa_Tra_{T[m]}"), srv[f"Usa_Tra_{T[m]}"]
    r_val[f"Compra_{T[m]}"], s_val[f"Compra_{T[m]}"] = valor_rep(rep, f"Compra_{T[m]}"), srv[f"Compra_{T[m]}"]
    r_val[f"Inv_Fin_{T[m]}"] = valor_rep(rep, f"Inv_Fin_{T[m]}")
    if m == MES_ACTUAL:
        r_val[f"Consumo_{T[m]}_Real_Hoy"], s_val[f"Consumo_{T[m]}_Real_Hoy"] = real_hoy_r, real_hoy_s
    r_val[f"Consumo_Inv_{T[m]}"] = real_hoy_r + valor_rep(rep, f"Pend_{T[m]}")
for m in MESES_REALES:
    r_val[f"Consumo_Inv_{T[m]}"] = valor_rep(rep, f"Real_{T[m]}")
medidas = sorted(set(r_val.columns) | set(s_val.columns) - set(GRUPO), key=lambda c: c)
medidas = [c for c in medidas if c not in GRUPO]
g = por_grupo(r_val, [c for c in medidas if c in r_val]).add(por_grupo(s_val, [c for c in medidas if c in s_val]), fill_value=0.0)
todos = SALDO_GRUPO.index.union(g.index)
g = g.reindex(todos).fillna(0.0)
g = g.join(SALDO_GRUPO.reindex(todos).fillna(0.0))
# Grupos sin presupuesto, saldo, consumo ni inventario no se muestran
g = g[g.abs().sum(axis=1) > 0.005]
g = g[~((g["Presupuesto"] == 0) & (g["Saldo"].abs() < 0.005) & (g.drop(columns=["Disponible_Odoo"]).abs().sum(axis=1) < 0.005))]
aj = aj_grupo.set_index(GRUPO)[list(COL_AJ.values())] if len(aj_grupo) else pd.DataFrame(columns=list(COL_AJ.values()))
for m, col in COL_AJ.items():
    g[col] = aj[col].reindex(g.index).fillna(0.0).values if len(aj) else 0.0
gasto_real = GASTO_REAL_MES.unstack("Mes", fill_value=0.0).reindex(g.index, fill_value=0.0)
for m in range(1, 13):
    g[f"Gasto_Real_{T[m]}"] = gasto_real[m].values if m in gasto_real else 0.0
g = g.reset_index()


def armar_resumen(g):
    d = pd.DataFrame(g[GRUPO])
    d["Presupuesto"] = g["Presupuesto"]
    d["Saldo_Odoo"] = g["Disponible_Odoo"]
    d["Saldo_Real_Hoy"] = g["Saldo"]
    d[f"Inv_Inicio_{T[MESES_CIERRE[0]]}"] = g["Inv_Inicio"]
    d["Inv_Hoy"] = g["Inv_Hoy"]
    for m in MESES_REALES:
        d[f"Consumo_{T[m]}_Real"] = g[f"Consumo_{T[m]}_Real"]
        d[f"Consumo_{T[m]}_Prueba"] = g[f"Consumo_{T[m]}_Prueba"]
        real = g[f"Consumo_{T[m]}_Real"]
        d[f"Desvio_Proy_{T[m]}"] = np.where(real > 0, g[f"Consumo_{T[m]}_Prueba"] / real.where(real > 0, 1) - 1, np.nan)
    for m in MESES_FUTUROS:
        d[f"Consumo_{T[m]}"] = g[f"Consumo_{T[m]}"]
    d["Consumo_Pendiente"] = g[[f"Pend_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
    d["Cubre_Inventario"] = g[[f"Usa_Inv_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
    d["Cubre_Transito"] = g[[f"Usa_Tra_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
    d["Compra_Nueva"] = g[[f"Compra_{T[m]}" for m in MESES_FUTUROS]].sum(axis=1)
    d["Ajustes"] = g[list(COL_AJ.values())].sum(axis=1)
    d["Saldo_31DIC"] = d["Saldo_Real_Hoy"] - d["Compra_Nueva"] - d["Ajustes"]
    d["Inv_31DIC"] = g["Inv_31DIC"]
    ini = d[f"Inv_Inicio_{T[MESES_CIERRE[0]]}"]
    d["Reduccion_Inv"] = np.where(ini > 0, 1 - d["Inv_31DIC"] / ini.where(ini > 0, 1), np.nan)
    d["Transito_Post_Cierre"] = g["Transito_Post_Cierre"]
    d["Estado"] = np.where(d["Saldo_31DIC"] < -UMBRAL_JUSTO, "FALTA", np.where(d["Saldo_31DIC"] > UMBRAL_JUSTO, "SOBRA", "JUSTO"))
    return d


ORDEN_EMPRESA = {n: i for i, n in enumerate(EMPRESAS)}
sumables = [c for c in g.columns if c not in GRUPO]
niv_centro = g.groupby(["Empresa", "Cuenta Analítica"])[sumables].sum().reset_index().assign(Situación="TOTAL CENTRO")
niv_empresa = g.groupby(["Empresa"])[sumables].sum().reset_index().assign(**{"Cuenta Analítica": "TOTAL EMPRESA", "Situación": ""})
gg = pd.concat([niv_empresa.assign(Nivel=1), niv_centro.assign(Nivel=2), g.assign(Nivel=3)], ignore_index=True)
gg["_e"] = gg["Empresa"].map(ORDEN_EMPRESA)
gg["_c"] = np.where(gg["Nivel"] == 1, "", gg["Cuenta Analítica"])
gg = gg.sort_values(["_e", "_c", "Nivel", "Situación"]).drop(columns=["_e", "_c"]).reset_index(drop=True)
resumen = armar_resumen(gg)
resumen["Nivel"] = gg["Nivel"].values
pd.set_option("display.float_format", "{:,.0f}".format)
resumen[["Empresa", "Cuenta Analítica", "Situación", "Saldo_Real_Hoy", "Inv_Hoy", "Consumo_Pendiente", "Cubre_Inventario", "Cubre_Transito",
         "Compra_Nueva", "Saldo_31DIC", "Inv_31DIC", "Estado"]]

##### 7. Flujo mensual, seguimiento y acciones

**Flujo mensual** por grupo (y subtotales), meses en columnas:
- **Saldo:** inicio de octubre = saldo de hoy + lo gastado (RQ) desde octubre. Gasto del mes: real (RQ creados en el mes) y, en el mes en curso y siguientes, + compra nueva + ajustes.
- **Inventario:** inicio, ingresos (compras, llegadas y otros movimientos), salidas por consumo y cierre. Meses reales con el kardex; los siguientes, con la cobertura proyectada.

**Seguimiento mensual:** gasto de enero a diciembre por grupo (real por mes del RQ; mes en curso real + proyección; meses siguientes proyección).

**Acciones** (una sola hoja, filtrar por `Accion`):
- `1 COMPRAR CON SOBRANTE`: en grupos que sobran, llaves del mismo grupo que terminan el año bajo su punto de reposición (A‑B y X‑Y primero), hasta agotar el sobrante.
- `2 POSTERGAR POR FALTA`: en grupos que faltan, compras nuevas de menor prioridad (C y Z primero) hasta cubrir el faltante.
- `3 NO COMPRAR: CONSUMIR STOCK`: sin rotación o sobrestock.
- `4 REVISAR: NO LLEGA AL CIERRE`: compra nueva cuyo lead time pasa el 31‑DIC.

In [ ]:
def bloque_flujo(fila):
    gasto_real_desde = sum(fila[f"Gasto_Real_{T[m]}"] for m in MESES_CIERRE if m <= MES_ACTUAL)
    saldo = fila["Saldo"] + gasto_real_desde
    inv = fila["Inv_Inicio"]
    c = {k: {} for k in ["Saldo presupuesto al inicio", "(-) Gasto del mes (RQ y compras)", "Saldo presupuesto al cierre",
                         "Consumo del mes (almacén + servicios)", "Inventario al inicio", "(+) Ingresos al almacén",
                         "(-) Salidas por consumo", "Inventario al cierre"]}
    for m in MESES_CIERRE:
        gasto = fila[f"Gasto_Real_{T[m]}"] if m <= MES_ACTUAL else 0.0
        if m in MESES_FUTUROS:
            gasto += fila[f"Compra_{T[m]}"] + fila[COL_AJ[m]]
        consumo = fila[f"Consumo_{T[m]}_Real"] if m in MESES_REALES else fila[f"Consumo_{T[m]}"]
        cons_inv = fila[f"Consumo_Inv_{T[m]}"]
        fin = fila[f"Inv_Fin_{T[m]}"]
        c["Saldo presupuesto al inicio"][m] = saldo
        c["(-) Gasto del mes (RQ y compras)"][m] = gasto
        saldo -= gasto
        c["Saldo presupuesto al cierre"][m] = saldo
        c["Consumo del mes (almacén + servicios)"][m] = consumo
        c["Inventario al inicio"][m] = inv
        c["(+) Ingresos al almacén"][m] = fin - inv + cons_inv
        c["(-) Salidas por consumo"][m] = cons_inv
        c["Inventario al cierre"][m] = fin
        inv = fin
    tiene_inv = any(abs(v) > 0.005 for k in ["Inventario al inicio", "Inventario al cierre", "(-) Salidas por consumo"] for v in c[k].values())
    return [(k, v) for k, v in c.items() if tiene_inv or not k.startswith(("Inventario", "(+) Ingresos", "(-) Salidas"))]


ETIQUETA = {m: f"{T[m]}_Real" if m in MESES_REALES else (f"{T[m]}_Real_Hoy+Proy" if m == MES_ACTUAL else f"{T[m]}_Proy") for m in MESES_CIERRE}
filas = []
for _, fila in gg.iterrows():
    for concepto, valores in bloque_flujo(fila):
        filas.append({"Empresa": fila["Empresa"], "Cuenta Analítica": fila["Cuenta Analítica"], "Situación": fila["Situación"], "Concepto": concepto,
                      **{ETIQUETA[m]: valores[m] for m in MESES_CIERRE}, "Nivel": fila["Nivel"]})
flujo = pd.DataFrame(filas)

ETQ_SEG = {m: f"{T[m]}_Real" if m < MES_ACTUAL else (f"{T[m]}_Real+Proy" if m == MES_ACTUAL else f"{T[m]}_Proy") for m in range(1, 13)}
seguimiento = pd.DataFrame(gg[GRUPO])
seguimiento["Presupuesto"] = gg["Presupuesto"]
for m in range(1, 13):
    v = gg[f"Gasto_Real_{T[m]}"] if m <= MES_ACTUAL else 0.0
    if m in MESES_FUTUROS:
        v = v + gg[f"Compra_{T[m]}"] + gg[COL_AJ[m]]
    seguimiento[ETQ_SEG[m]] = v
seguimiento["Gasto_Anio"] = seguimiento[list(ETQ_SEG.values())].sum(axis=1)
seguimiento["Saldo_31DIC"] = seguimiento["Presupuesto"] - seguimiento["Gasto_Anio"]
seguimiento["Ejecucion"] = np.where(seguimiento["Presupuesto"] > 0, seguimiento["Gasto_Anio"] / seguimiento["Presupuesto"].where(seguimiento["Presupuesto"] > 0, 1), np.nan)
seguimiento["Nivel"] = gg["Nivel"].values

"""
    ACCIONES
"""
PRIO_ABC, PRIO_XYZ = {"A": 0, "B": 1, "C": 2}, {"X": 0, "Y": 1, "Z": 2}
COLS_ACC = ["Accion", "Empresa", "Cuenta Analítica", "Situación", "Llave", "Producto", "ABC_XYZ", "Estado", "Precio", "Und", "Monto", "Acumulado", "Motivo"]
estado_grupo = resumen[resumen["Nivel"] == 3].set_index(GRUPO)["Saldo_31DIC"]
acciones = []

cand = rep[(rep["Prom_12M"] > 0) & (rep["Precio"] > 0)].copy()
cand["Und"] = np.ceil((cand["Punto_Reposicion"] - cand["Inv_31DIC"]).clip(lower=0))
cand = cand[cand["Und"] > 0]
cand["Monto"] = cand["Und"] * cand["Precio"]
cand["_p"] = cand["ABC"].map(PRIO_ABC).fillna(3) * 3 + cand["XYZ"].map(PRIO_XYZ).fillna(3)
for clave, sobra in estado_grupo[estado_grupo > UMBRAL_JUSTO].items():
    c = cand[(cand[GRUPO] == pd.Series(clave, index=GRUPO)).all(axis=1)].sort_values(["_p", "Monto"], ascending=[True, False])
    acum = 0.0
    for _, r in c.iterrows():
        if acum + r["Monto"] > sobra:
            continue
        acum += r["Monto"]
        acciones.append({**{k: r[k] for k in GRUPO + ["Llave", "Producto", "ABC_XYZ", "Estado", "Precio", "Und", "Monto"]}, "Acumulado": acum,
                         "Accion": "1 COMPRAR CON SOBRANTE",
                         "Motivo": f"Termina el año con {r['Inv_31DIC']:,.1f} und, bajo su punto de reposición {r['Punto_Reposicion']:,.1f} · sobrante del grupo S/ {sobra:,.0f}"})

post = rep[rep["Compra_Nueva"] > 0].copy()
post["_p"] = -(post["ABC"].map(PRIO_ABC).fillna(3) * 3 + post["XYZ"].map(PRIO_XYZ).fillna(3))
for clave, falta in estado_grupo[estado_grupo < -UMBRAL_JUSTO].items():
    c = post[(post[GRUPO] == pd.Series(clave, index=GRUPO)).all(axis=1)].sort_values(["_p", "Compra_Nueva"], ascending=[True, True])
    acum = 0.0
    for _, r in c.iterrows():
        if acum >= -falta:
            break
        acum += r["Compra_Nueva"]
        acciones.append({**{k: r[k] for k in GRUPO + ["Llave", "Producto", "ABC_XYZ", "Estado", "Precio"]}, "Und": r["Compra_Nueva_Und"], "Monto": r["Compra_Nueva"],
                         "Acumulado": acum, "Accion": "2 POSTERGAR POR FALTA",
                         "Motivo": f"Compra nueva de prioridad baja ({r['ABC_XYZ']}) · faltante del grupo S/ {-falta:,.0f}"})

for _, r in rep[rep["Estado"].isin(["SIN ROTACIÓN", "SOBRESTOCK"]) & (rep["Stock_Hoy"] > 0)].sort_values("Stock_Hoy", ascending=False).iterrows():
    acciones.append({**{k: r[k] for k in GRUPO + ["Llave", "Producto", "ABC_XYZ", "Estado", "Precio"]}, "Und": r["Stock_Hoy"], "Monto": r["Stock_Hoy"] * r["Precio"],
                     "Acumulado": np.nan, "Accion": "3 NO COMPRAR: CONSUMIR STOCK",
                     "Motivo": "Sin consumo en 12 meses" if r["Estado"] == "SIN ROTACIÓN" else f"Al 31-DIC cubre {r['Cobertura_31DIC_Meses']:,.1f} meses"})

for _, r in rep[rep["Riesgo_Cierre"] != ""].iterrows():
    acciones.append({**{k: r[k] for k in GRUPO + ["Llave", "Producto", "ABC_XYZ", "Estado", "Precio"]}, "Und": r["Compra_Nueva_Und"], "Monto": r["Compra_Nueva"],
                     "Acumulado": np.nan, "Accion": "4 REVISAR: NO LLEGA AL CIERRE", "Motivo": r["Riesgo_Cierre"]})
acciones = pd.DataFrame(acciones, columns=COLS_ACC)
acciones["_m"] = acciones["Monto"].fillna(0.0)
acciones = acciones.sort_values(["Accion", "Empresa", "Cuenta Analítica", "Situación"]).drop(columns="_m").reset_index(drop=True)
print(acciones.groupby(["Accion", "Empresa"]).agg(Filas=("Llave", "size"), Monto=("Monto", "sum")).round(0).to_string())

##### 8. Guardar el archivo de ajustes y el resultado

- **`Ajustes_Proyeccion_2026.xlsx`** (se edita): INSTRUCCIONES, AJUSTES_GRUPO (todos los grupos), REPUESTOS (clase A, B‑Z y los ya corregidos; se pueden agregar filas) y SERVICIOS. Columnas amarillas = editables. Si el archivo existe, se guarda un respaldo `*_respaldo.xlsx`. Si está abierto en Excel no se actualiza (se avisa) y el resultado igual se genera con lo leído.
- **`Proyeccion_Presupuesto_2026.xlsx`**: RESUMEN, FLUJO_MENSUAL, SEGUIMIENTO_MENSUAL, DETALLE_REPUESTOS, DETALLE_SERVICIOS, ACCIONES. Los totales por centro y empresa van en negrita y con fondo; en RESUMEN las situaciones se pueden plegar bajo su centro (botones 1‑2 a la izquierda).

Diseño: sin cuadrícula, Consolas 8, soles con negativos en rojo, encabezados por color (azul = identificación, verde = presupuesto y saldo, ámbar = real, morado = proyección, azul acero = inventario y tránsito, rojo = compra nueva y estado, naranja = editable, gris = notas).

In [ ]:
FUENTE, FUENTE_T, FUENTE_N = Font(name="Consolas", size=8), Font(name="Consolas", size=8, bold=True, color="FFFFFF"), Font(name="Consolas", size=8, bold=True)
F_SOLES, F_CANT, F_PCT, F_DIAS = '"S/" #,##0.00;[Red]-"S/" #,##0.00', "#,##0.00;[Red]-#,##0.00", "0.0%", "#,##0"
COLOR = {"id": "1F4E78", "saldo": "375623", "real": "BF8F00", "plan": "7030A0", "inv": "2E75B6", "def": "C00000", "nota": "595959", "dec": "C65911"}
FONDO = {1: PatternFill("solid", fgColor="D9E1F2"), 2: PatternFill("solid", fgColor="EDEDED")}
AMARILLO = PatternFill("solid", fgColor="FFF2CC")
TEXTO = {"Producto": 34, "Cuenta Analítica": 30, "Situación": 34, "Concepto": 34, "Motivo": 60, "Metodo": 30, "Comentario": 30,
         "Comentario_Usuario": 26, "Familia": 18, "Texto": 130, "Riesgo_Cierre": 28}
CENTRADAS = {"Empresa", "Llave", "Cod_Producto", "ABC", "XYZ", "ABC_XYZ", "Estado", "Categoria", "Tipo", "Accion", "UdM"}
EDITABLES = set(COL_AJ.values()) | set(COL_CORR.values()) | {"Comentario"}


def grupo_col(c):
    if c in EDITABLES:
        return "dec"
    if c[:3] in NOMBRE_MES.values():
        return "plan" if "Proy" in c else "real"
    if c.startswith(("Presupuesto", "Saldo", "Precio", "Gasto", "Ejecucion", "Ejecutado")):
        return "saldo"
    if c.startswith(("Inv_", "Stock", "RQ_", "Transito", "Cubre", "Usa_", "Llega", "Cobertura", "Lead", "Punto", "Reduccion")):
        return "inv"
    if c.startswith(("Compra", "Ajustes", "Estado", "Riesgo", "Monto", "Accion", "Und", "Acumulado")):
        return "def"
    if "Real" in c or c.startswith(("Desvio", "Prueba")):
        return "real"
    if c.startswith(("Consumo", "Proy", "Sug", "Pend", "Prom", "Desv_", "CV", "Valor_12M", "Meses_", "ABC", "XYZ", "Metodo", "Tipo", "Categoria")):
        return "plan"
    if c.startswith(("Motivo", "Texto", "Comentario_Usuario")):
        return "nota"
    return "id"


def formato_col(c, df, soles):
    if c.startswith(("Desvio", "Reduccion", "Ejecucion")):
        return F_PCT
    if c.startswith(("Lead_Time", "Meses_")):
        return F_DIAS
    if soles or c.startswith(("Presupuesto", "Saldo", "Precio", "Monto", "Acumulado", "Inv_", "Consumo", "Compra_Nueva", "Cubre", "Ajust",
                              "Gasto", "Transito_Post", "Ejecutado", "Valor_12M")) or c[:3] in NOMBRE_MES.values():
        return F_SOLES
    return F_CANT


def ancho(df, c, fmt):
    if c in TEXTO:
        return TEXTO[c]
    v = df[c].replace([np.inf, -np.inf], np.nan).dropna()
    if fmt == F_SOLES:
        largo = max([len(f"S/ {x:,.2f}") + 1 for x in v], default=0)
    elif fmt in (F_CANT, F_DIAS):
        largo = max([len(f"{x:,.2f}") for x in v], default=0)
    elif fmt == F_PCT:
        largo = 7
    else:
        largo = max([len(str(x)) for x in v], default=0)
    return min(max(largo, len(c) * 0.85, 6) + 2, 28)


def dar_formato(hoja, df, soles_por_defecto=False, niveles=None, plegar=False, negrita=None):
    hoja.sheet_view.showGridLines = False
    fmt, ali = {}, {}
    for i, c in enumerate(df.columns, start=1):
        celda = hoja.cell(1, i)
        celda.font, celda.border = FUENTE_T, Border()
        celda.fill = PatternFill("solid", fgColor=COLOR[grupo_col(c)])
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        num = pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])
        if num:
            fmt[i] = formato_col(c, df, soles_por_defecto and c not in set(COL_CORR.values()) | {"Und", "Lead_Time_Dias"})
        ali[i] = Alignment(horizontal="right" if num else "center" if c in CENTRADAS else "left", vertical="center")
        hoja.column_dimensions[get_column_letter(i)].width = ancho(df, c, fmt.get(i))
    editables = {i for i, c in enumerate(df.columns, start=1) if c in EDITABLES}
    for r, fila in enumerate(hoja.iter_rows(min_row=2), start=0):
        nivel = niveles[r] if niveles is not None else 3
        fuerte = nivel < 3 or (negrita is not None and negrita[r])
        for celda in fila:
            celda.font = FUENTE_N if fuerte else FUENTE
            celda.alignment = ali[celda.column]
            if nivel in FONDO:
                celda.fill = FONDO[nivel]
            elif celda.column in editables:
                celda.fill = AMARILLO
            if celda.column in fmt:
                celda.number_format = fmt[celda.column]
        if plegar and nivel == 3:
            hoja.row_dimensions[r + 2].outlineLevel = 1
    if plegar:
        hoja.sheet_properties.outlinePr.summaryBelow = False
    hoja.row_dimensions[1].height = 30
    hoja.freeze_panes = "D2" if "Situación" in df.columns[:3] else "A2"
    hoja.auto_filter.ref = hoja.dimensions


def escribir(archivo, hojas):
    with pd.ExcelWriter(archivo, engine="openpyxl") as w:
        for nombre, spec in hojas.items():
            df = spec["df"].replace([np.inf, -np.inf], np.nan)
            df.to_excel(w, sheet_name=nombre, index=False)
            dar_formato(w.sheets[nombre], df, spec.get("soles", False), spec.get("niveles"), spec.get("plegar", False), spec.get("negrita"))


"""
    1. ARCHIVO DE AJUSTES (SE CONSERVA LO EDITADO)
"""
res3 = resumen[resumen["Nivel"] == 3]
hoja_grupo = res3[GRUPO + ["Saldo_Real_Hoy", "Inv_Hoy"] + [f"Consumo_{T[m]}" for m in MESES_FUTUROS] + ["Compra_Nueva", "Saldo_31DIC"]].copy()
hoja_grupo = hoja_grupo.merge(aj_grupo, on=GRUPO, how="left")
for c in list(COL_AJ.values()) + ["Comentario"]:
    hoja_grupo[c] = hoja_grupo[c] if c in hoja_grupo else np.nan

marcar = rep["ABC"].eq("A") | (rep["ABC"].eq("B") & rep["XYZ"].eq("Z"))
claves_aj = set(zip(aj_rep["Empresa"], aj_rep["Llave"]))
hoja_rep = rep[marcar | pd.Series([k in claves_aj for k in zip(rep["Empresa"], rep["Llave"])], index=rep.index)].copy()
hoja_rep = hoja_rep[GRUPO + ["Llave", "Producto", "UdM", "ABC_XYZ", "Metodo", "Precio", "Prom_6M", "Prom_12M"] + [f"Real_{T[m]}" for m in MESES_CIERRE if m <= MES_ACTUAL]
                    + ["Stock_Hoy", "RQ_COC", "RQ_SOC"] + [f"Sug_{T[m]}" for m in MESES_FUTUROS] + list(COL_CORR.values()) + ["Comentario_Usuario"]]
hoja_rep = hoja_rep.rename(columns={"Comentario_Usuario": "Comentario"})
sin_llave = aj_rep[[k not in set(zip(rep["Empresa"], rep["Llave"])) for k in zip(aj_rep["Empresa"], aj_rep["Llave"])]]
if len(sin_llave):
    print(f"AVISO: {len(sin_llave)} correcciones de REPUESTOS con llave fuera del alcance (se conservan sin usar):", sin_llave["Llave"].tolist()[:10])
    hoja_rep = pd.concat([hoja_rep, sin_llave.assign(Producto="(llave fuera del alcance: revisar)")], ignore_index=True)
hoja_rep = hoja_rep.sort_values(["Empresa", "Situación", "ABC_XYZ", "Prom_12M"], ascending=[True, True, True, False], na_position="last")

hoja_srv = srv[GRUPO + ["Cod_Producto", "Producto", "Tipo", "Presupuesto", "Ejecutado_Anio", "Prom_6M", "Meses_Facturados_6M"]
               + [f"Real_{T[m]}" for m in MESES_CIERRE if m <= MES_ACTUAL] + ["Transito"] + [f"Sug_{T[m]}" for m in MESES_FUTUROS]
               + list(COL_CORR.values()) + ["Comentario_Usuario"]].rename(columns={"Comentario_Usuario": "Comentario"})
hoja_srv = hoja_srv.sort_values(["Empresa", "Cuenta Analítica", "Situación", "Prom_6M"], ascending=[True, True, True, False])

meses_txt = ", ".join(T[m] for m in MESES_FUTUROS)
instrucciones = pd.DataFrame({"Texto": [
    f"AJUSTES DE LA PROYECCIÓN {ANIO} · actualizado {HOY:%d/%m/%Y}",
    "Editar SOLO las columnas amarillas. Celda vacía = usar el cálculo. Guardar, cerrar y volver a ejecutar Proyeccion_Presupuesto.ipynb.",
    f"AJUSTES_GRUPO: Ajuste_ de {meses_txt} en soles por grupo. Positivo = gasto adicional que el cálculo no ve (campaña, compra puntual); negativo = menos gasto.",
    f"REPUESTOS: Corr_ de {meses_txt} = unidades TOTALES del mes para esa llave (reemplaza la sugerencia Sug_). En el mes en curso se descuenta lo ya consumido.",
    "REPUESTOS: vienen los A y los B-Z (valiosos). Para corregir otro, agregar una fila con Empresa y Llave (o el código del producto) y su Corr_.",
    f"SERVICIOS: Corr_ de {meses_txt} = monto en soles con IGV del mes para ese servicio y centro. EVENTUAL / SIN EJECUCIÓN vienen en 0: decidir aquí.",
    "Las filas corregidas se conservan al volver a ejecutar; se guarda un respaldo *_respaldo.xlsx del archivo anterior.",
]})
try:
    if os.path.exists(ARCHIVO_AJUSTES):
        shutil.copyfile(ARCHIVO_AJUSTES, ARCHIVO_AJUSTES.replace(".xlsx", "_respaldo.xlsx"))
    escribir(ARCHIVO_AJUSTES, {"INSTRUCCIONES": {"df": instrucciones},
                               "AJUSTES_GRUPO": {"df": hoja_grupo, "soles": True},
                               "REPUESTOS": {"df": hoja_rep},
                               "SERVICIOS": {"df": hoja_srv, "soles": True}})
    print(f"Archivo de ajustes actualizado: {ARCHIVO_AJUSTES} (grupos {len(hoja_grupo)}, repuestos {len(hoja_rep)}, servicios {len(hoja_srv)})")
except PermissionError:
    print(f"AVISO: {ARCHIVO_AJUSTES} está abierto; ciérralo para actualizarlo. El resultado se genera con los ajustes leídos.")

"""
    2. RESULTADO
"""
if EXPORTAR_EXCEL:
    COLS_REP = (GRUPO + ["Llave", "Producto", "Familia", "UdM", "Categoria", "ABC", "XYZ", "ABC_XYZ", "CV", "Valor_12M", "Meses_Con_Consumo_12M", "Prom_6M", "Prom_12M",
                         "Metodo", "Precio", "Lead_Time_Dias", "Stock_Inicio"] + [f"Stock_{T[m]}" for m in MESES_REALES] + ["Stock_Hoy", "RQ_COC", "RQ_SOC"]
                + [c for m in MESES_REALES for c in (f"Real_{T[m]}", f"Prueba_{T[m]}")]
                + [c for m in MESES_FUTUROS for c in ([f"Real_{T[m]}"] if m == MES_ACTUAL else []) + [f"Sug_{T[m]}", COL_CORR[m], f"Proy_{T[m]}", f"Pend_{T[m]}",
                                                                                                       f"Usa_Inv_{T[m]}", f"Llega_{T[m]}", f"Usa_Tra_{T[m]}", f"Compra_{T[m]}", f"Inv_Fin_{T[m]}"]]
                + ["Compra_Nueva_Und", "Compra_Nueva", "Inv_31DIC", "Transito_Post_Cierre", "Cobertura_31DIC_Meses", "Punto_Reposicion", "Estado", "Riesgo_Cierre",
                   "Comentario_Usuario"])
    COLS_SRV = (GRUPO + ["Cod_Producto", "Producto", "Tipo", "Presupuesto", "Ejecutado_Anio", "Meses_Facturados_6M", "Prom_6M"]
                + [c for m in MESES_REALES for c in (f"Real_{T[m]}", f"Prueba_{T[m]}")]
                + [c for m in MESES_FUTUROS for c in ([f"Real_{T[m]}"] if m == MES_ACTUAL else []) + [f"Sug_{T[m]}", COL_CORR[m], f"Proy_{T[m]}", f"Pend_{T[m]}",
                                                                                                       f"Usa_Tra_{T[m]}", f"Compra_{T[m]}"]]
                + ["RQ_COC", "RQ_SOC", "Compra_Nueva", "Comentario_Usuario"])
    escribir(ARCHIVO_EXCEL, {
        "RESUMEN": {"df": resumen.drop(columns="Nivel"), "soles": True, "niveles": list(resumen["Nivel"]), "plegar": True},
        "FLUJO_MENSUAL": {"df": flujo.drop(columns="Nivel"), "soles": True, "niveles": list(flujo["Nivel"]),
                          "negrita": list(flujo["Concepto"].str.contains("al cierre"))},
        "SEGUIMIENTO_MENSUAL": {"df": seguimiento.drop(columns="Nivel"), "soles": True, "niveles": list(seguimiento["Nivel"]), "plegar": True},
        "DETALLE_REPUESTOS": {"df": rep[COLS_REP].sort_values(GRUPO + ["Compra_Nueva"], ascending=[True, True, True, False])},
        "DETALLE_SERVICIOS": {"df": srv[COLS_SRV].sort_values(GRUPO + ["Compra_Nueva"], ascending=[True, True, True, False]), "soles": True},
        "ACCIONES": {"df": acciones},
    })
    print(f"Resultado generado: {ARCHIVO_EXCEL}")